In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
import numpy as np
import torch.nn as nn
from src.data import (load_fd001, scale_features,
                      make_train_windows, make_test_windows)
from src.models import LSTMRegressor
from src.train import train_model, predict, rmse


In [6]:
WINDOW = 30
train, test, test_rul, feats = load_fd001()
train, test, _ = scale_features(train, test, feats)

X_all, y_all, units = make_train_windows(train, feats, WINDOW)
X_test = make_test_windows(test, feats, WINDOW)
y_test = np.minimum(test_rul, 125)

# hold out 20 training engines for validation (split by engine, not by row)
rng = np.random.default_rng(42)
val_units = rng.choice(np.unique(units), size=20, replace=False)
val_mask = np.isin(units, val_units)
X_tr, y_tr = X_all[~val_mask], y_all[~val_mask]
X_val, y_val = X_all[val_mask], y_all[val_mask]
print(X_tr.shape, X_val.shape, X_test.shape)

model = LSTMRegressor(n_features=len(feats), out_bias=float(y_tr.mean()))
model, hist = train_model(model, X_tr, y_tr, X_val, y_val,
                          loss_fn=nn.MSELoss(), epochs=60)
print("Test RMSE:", rmse(y_test, predict(model, X_test)))

(14070, 30, 17) (3661, 30, 17) (100, 30, 17)
epoch  1  train loss  812.689  val RMSE  21.66
epoch  2  train loss  364.672  val RMSE  17.64
epoch  3  train loss  249.715  val RMSE  16.26
epoch  4  train loss  228.426  val RMSE  15.82
epoch  5  train loss  227.034  val RMSE  15.67
epoch  6  train loss  202.031  val RMSE  15.48
epoch  7  train loss  201.636  val RMSE  15.51
epoch  8  train loss  191.907  val RMSE  15.01
epoch  9  train loss  189.361  val RMSE  16.25
epoch 10  train loss  191.999  val RMSE  14.84
epoch 11  train loss  188.161  val RMSE  14.83
epoch 12  train loss  181.657  val RMSE  14.91
epoch 13  train loss  178.553  val RMSE  14.71
epoch 14  train loss  178.201  val RMSE  14.70
epoch 15  train loss  176.371  val RMSE  15.32
epoch 16  train loss  172.764  val RMSE  14.83
epoch 17  train loss  177.980  val RMSE  14.80
epoch 18  train loss  167.482  val RMSE  14.60
epoch 19  train loss  169.643  val RMSE  14.86
epoch 20  train loss  165.030  val RMSE  14.49
epoch 21  train